# Maritime Speed Prediction - Baseline ML Models

This notebook implements initial ML models to predict ship speed (STW_kn) based on weather and operational conditions. The analysis includes baseline models, XGBoost, cross-validation, and feature importance analysis to identify which weather conditions have the biggest impact on ship speed.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/MasterSet_features.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names and types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nTarget variable (STW_kn) statistics:")
print(df['STW_kn'].describe())

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['STW_kn'].dropna(), bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('STW_kn (Ship Through Water Speed)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ship Speed')
axes[1].boxplot(df['STW_kn'].dropna(), vert=True)
axes[1].set_ylabel('STW_kn (knots)')
axes[1].set_title('Box Plot of Ship Speed')
plt.tight_layout()
plt.show()

## 3. Prepare Data for Modeling

In [ ]:
# Define features (weather and operational conditions)
# Exclude columns that are not features or are identifiers
exclude_cols = ['Time', 'Lat', 'Lon', 'Speed_kn', 'Calc_Speed_kn', 'Course_deg', 
                'Dist_since_last_nm', 'Total_dist_nm', 'Voyage_ID', 'STW_kn', 'STH_deg']

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Selected features ({len(feature_cols)}):")
print(feature_cols)

# Target variable
target = 'STW_kn'

# Create feature matrix and target vector, dropping NaN values
df_clean = df[feature_cols + [target]].dropna()
print(f"\nDataset after removing NaN: {df_clean.shape}")

X = df_clean[feature_cols].values
y = df_clean[target].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Scale features for better model performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nFeatures scaled successfully!")
print(f"X_scaled shape: {X_scaled.shape}")

## 4. Split Data with KFold Cross-Validation

In [ ]:
# Initialize KFold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("KFold Cross-Validation Setup:")
print(f"Number of folds: {kfold.get_n_splits()}")
print(f"Total samples: {len(X_scaled)}")

# Display fold information
fold_info = []
for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_scaled), 1):
    fold_info.append({
        'Fold': fold_idx,
        'Train samples': len(train_idx),
        'Test samples': len(test_idx),
        'Train%': f"{len(train_idx)/len(X_scaled)*100:.1f}%",
        'Test%': f"{len(test_idx)/len(X_scaled)*100:.1f}%"
    })

fold_df = pd.DataFrame(fold_info)
print("\n", fold_df.to_string(index=False))

## 5. Train Baseline Models

In [ ]:
# Define baseline models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Store results
baseline_results = {}
scoring = ['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error']

print("Training baseline models with KFold cross-validation...\n")

for model_name, model in models.items():
    print(f"Training {model_name}...")
    
    # Cross-validation
    cv_results = cross_validate(model, X_scaled, y, cv=kfold, 
                                scoring=scoring, return_train_score=True)
    
    # Calculate metrics
    mae_scores = -cv_results['test_neg_mean_absolute_error']
    rmse_scores = np.sqrt(-cv_results['test_neg_mean_squared_error'])
    r2_scores = cv_results['test_r2']
    
    baseline_results[model_name] = {
        'MAE': mae_scores,
        'RMSE': rmse_scores,
        'R2': r2_scores,
        'MAE_mean': mae_scores.mean(),
        'MAE_std': mae_scores.std(),
        'RMSE_mean': rmse_scores.mean(),
        'RMSE_std': rmse_scores.std(),
        'R2_mean': r2_scores.mean(),
        'R2_std': r2_scores.std()
    }
    
    print(f"  MAE: {mae_scores.mean():.4f} (+/- {mae_scores.std():.4f})")
    print(f"  RMSE: {rmse_scores.mean():.4f} (+/- {rmse_scores.std():.4f})")
    print(f"  R²: {r2_scores.mean():.4f} (+/- {r2_scores.std():.4f})\n")

print("Baseline models training completed!")

## 6. Train XGBoost Model

In [ ]:
# Initialize XGBoost model with hyperparameters
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

print("Training XGBoost model with KFold cross-validation...\n")

# Cross-validation for XGBoost
xgb_cv_results = cross_validate(xgb_model, X_scaled, y, cv=kfold, 
                                scoring=scoring, return_train_score=True)

# Calculate XGBoost metrics
xgb_mae_scores = -xgb_cv_results['test_neg_mean_absolute_error']
xgb_rmse_scores = np.sqrt(-xgb_cv_results['test_neg_mean_squared_error'])
xgb_r2_scores = xgb_cv_results['test_r2']

xgb_results = {
    'MAE': xgb_mae_scores,
    'RMSE': xgb_rmse_scores,
    'R2': xgb_r2_scores,
    'MAE_mean': xgb_mae_scores.mean(),
    'MAE_std': xgb_mae_scores.std(),
    'RMSE_mean': xgb_rmse_scores.mean(),
    'RMSE_std': xgb_rmse_scores.std(),
    'R2_mean': xgb_r2_scores.mean(),
    'R2_std': xgb_r2_scores.std()
}

print("XGBoost Model Performance:")
print(f"  MAE: {xgb_mae_scores.mean():.4f} (+/- {xgb_mae_scores.std():.4f})")
print(f"  RMSE: {xgb_rmse_scores.mean():.4f} (+/- {xgb_rmse_scores.std():.4f})")
print(f"  R²: {xgb_r2_scores.mean():.4f} (+/- {xgb_r2_scores.std():.4f})")

# Train final XGBoost model on full dataset for feature importance
print("\nTraining final XGBoost model on full dataset for feature importance...")
xgb_model.fit(X_scaled, y, verbose=0)
print("XGBoost training completed!")

## 7. Evaluate Model Performance

In [ ]:
# Create comprehensive evaluation summary
all_results = baseline_results.copy()
all_results['XGBoost'] = xgb_results

# Create summary table
summary_data = []
for model_name, metrics in all_results.items():
    summary_data.append({
        'Model': model_name,
        'MAE (mean)': f"{metrics['MAE_mean']:.4f}",
        'MAE (std)': f"{metrics['MAE_std']:.4f}",
        'RMSE (mean)': f"{metrics['RMSE_mean']:.4f}",
        'RMSE (std)': f"{metrics['RMSE_std']:.4f}",
        'R² (mean)': f"{metrics['R2_mean']:.4f}",
        'R² (std)': f"{metrics['R2_std']:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("MODEL PERFORMANCE SUMMARY (5-Fold Cross-Validation)")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

# Additional metrics: Calculate accuracy-like metric (% predictions within 1 knot)
print("\nAdditional Metrics - Prediction Accuracy (within 1 knot error):")
print("-" * 50)

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_scaled), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Make predictions with XGBoost
    xgb_temp = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
    )
    xgb_temp.fit(X_train, y_train)
    y_pred = xgb_temp.predict(X_test)
    
    # Calculate accuracy (% within 1 knot)
    accuracy = np.sum(np.abs(y_pred - y_test) <= 1.0) / len(y_test) * 100
    
    if fold_idx == 1:
        print(f"Fold {fold_idx}: {accuracy:.2f}% of predictions within ±1 knot")
    elif fold_idx == 2:
        print(f"Fold {fold_idx}: {accuracy:.2f}% of predictions within ±1 knot")
    elif fold_idx == 3:
        print(f"Fold {fold_idx}: {accuracy:.2f}% of predictions within ±1 knot")
    elif fold_idx == 4:
        print(f"Fold {fold_idx}: {accuracy:.2f}% of predictions within ±1 knot")
    else:
        print(f"Fold {fold_idx}: {accuracy:.2f}% of predictions within ±1 knot")

## 8. Feature Importance Analysis

In [ ]:
# Extract feature importance from XGBoost model
feature_importance = xgb_model.feature_importances_
feature_names = np.array(feature_cols)

# Sort by importance
importance_idx = np.argsort(feature_importance)[::-1]
sorted_importance = feature_importance[importance_idx]
sorted_names = feature_names[importance_idx]

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': sorted_names,
    'Importance': sorted_importance,
    'Normalized': sorted_importance / sorted_importance.sum()
})

print("\nTop 15 Most Important Features:")
print("-" * 60)
print(importance_df.head(15).to_string(index=False))

# Identify weather-related features
weather_features = ['u10', 'v10', 'H_s', 'mwd', 'mwp', 'fg10', 'uo', 'vo', 
                   'True_Wind_Speed_ms', 'AWS_ms', 'AWA_deg', 'AWA_abs_deg']
weather_importance_df = importance_df[importance_df['Feature'].isin(weather_features)]
weather_importance_df = weather_importance_df.sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("WEATHER IMPACT ON SHIP SPEED (Feature Importance)")
print("="*60)
print(weather_importance_df.to_string(index=False))
print(f"\nTotal weather feature importance: {weather_importance_df['Importance'].sum():.4f}")
print(f"Percentage of model: {weather_importance_df['Importance'].sum() / sorted_importance.sum() * 100:.2f}%")

## 9. Visualize Weather Impact on Ship Speed

In [ ]:
# Create scatter plots for key weather features vs ship speed
key_weather = {
    'u10': 'U Wind Component (m/s)',
    'v10': 'V Wind Component (m/s)',
    'H_s': 'Wave Height (m)',
    'True_Wind_Speed_ms': 'True Wind Speed (m/s)',
    'AWS_ms': 'Apparent Wind Speed (m/s)'
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (feature, label) in enumerate(key_weather.items()):
    ax = axes[idx]
    
    # Create scatter plot with alpha for density visualization
    scatter = ax.scatter(df_clean[feature], df_clean[target], 
                        alpha=0.3, s=10, c=df_clean[feature], cmap='viridis')
    
    ax.set_xlabel(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Ship Speed STW (knots)', fontsize=11, fontweight='bold')
    ax.set_title(f'Impact of {label} on Ship Speed', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label(label, fontsize=9)

# Remove extra subplot
axes[5].remove()

plt.tight_layout()
plt.savefig('../results/weather_impact_on_speed.png', dpi=300, bbox_inches='tight')
plt.show()

print("Weather impact scatter plots saved!")

In [ ]:
# Create correlation heatmap between weather features and ship speed
weather_cols = [col for col in feature_cols if col in weather_features]
correlation_df = df_clean[weather_cols + [target]].corr()

# Extract correlations with target
target_corr = correlation_df[target].drop(target).sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Correlation bar plot
colors = ['green' if x > 0 else 'red' for x in target_corr.values]
ax1.barh(target_corr.index, target_corr.values, color=colors, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Correlation with Ship Speed', fontsize=12, fontweight='bold')
ax1.set_title('Weather Features Correlation with STW_kn', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')
ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# Heatmap of weather-speed correlations
sns.heatmap(correlation_df.loc[weather_cols + [target], [target]], 
            annot=True, fmt='.3f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Correlation'}, ax=ax2, linewidths=0.5)
ax2.set_title('Weather Feature - Ship Speed Correlation Heatmap', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/weather_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Weather correlation analysis plots saved!")

## 10. Plot Model Results

In [ ]:
# 1. Feature Importance Bar Plot (Top 20 features)
fig, ax = plt.subplots(figsize=(12, 8))
top_n = 20
top_importance_idx = importance_idx[:top_n]
top_features = feature_names[top_importance_idx]
top_scores = feature_importance[top_importance_idx]

# Color code weather vs operational features
colors_list = ['#FF6B6B' if feat in weather_features else '#4ECDC4' for feat in top_features]

bars = ax.barh(range(len(top_features)), top_scores, color=colors_list, edgecolor='black', linewidth=1.2)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features)
ax.set_xlabel('Feature Importance Score', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Most Important Features for Ship Speed Prediction\n(Red: Weather, Blue: Operational)', 
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Add values on bars
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2, f'{width:.4f}', 
           ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/feature_importance_top20.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature importance plot saved!")

In [ ]:
# 2. Model Comparison - Performance Metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Extract model names and metrics
models_list = list(all_results.keys())
mae_means = [all_results[m]['MAE_mean'] for m in models_list]
mae_stds = [all_results[m]['MAE_std'] for m in models_list]
rmse_means = [all_results[m]['RMSE_mean'] for m in models_list]
rmse_stds = [all_results[m]['RMSE_std'] for m in models_list]
r2_means = [all_results[m]['R2_mean'] for m in models_list]
r2_stds = [all_results[m]['R2_std'] for m in models_list]

# Plot 1: MAE comparison
x_pos = np.arange(len(models_list))
axes[0].bar(x_pos, mae_means, yerr=mae_stds, capsize=5, alpha=0.8, 
           color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Mean Absolute Error (knots)', fontsize=11, fontweight='bold')
axes[0].set_title('MAE Comparison Across Models', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models_list, rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: RMSE comparison
axes[1].bar(x_pos, rmse_means, yerr=rmse_stds, capsize=5, alpha=0.8,
           color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('Root Mean Squared Error (knots)', fontsize=11, fontweight='bold')
axes[1].set_title('RMSE Comparison Across Models', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(models_list, rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: R² comparison
axes[2].bar(x_pos, r2_means, yerr=r2_stds, capsize=5, alpha=0.8,
           color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], edgecolor='black', linewidth=1.2)
axes[2].set_ylabel('R² Score', fontsize=11, fontweight='bold')
axes[2].set_title('R² Comparison Across Models', fontsize=12, fontweight='bold')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(models_list, rotation=45, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8)

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison plot saved!")

In [ ]:
# 3. Cross-Validation Fold Performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot R² scores across folds for all models
fold_numbers = np.arange(1, 6)
for model_name, metrics in all_results.items():
    ax1.plot(fold_numbers, metrics['R2'], marker='o', linewidth=2, 
            markersize=8, label=model_name)

ax1.set_xlabel('Fold Number', fontsize=12, fontweight='bold')
ax1.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax1.set_title('R² Score Across Cross-Validation Folds', fontsize=13, fontweight='bold')
ax1.set_xticks(fold_numbers)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='best', fontsize=10)

# Plot MAE scores across folds for all models
for model_name, metrics in all_results.items():
    ax2.plot(fold_numbers, metrics['MAE'], marker='s', linewidth=2, 
            markersize=8, label=model_name)

ax2.set_xlabel('Fold Number', fontsize=12, fontweight='bold')
ax2.set_ylabel('MAE (knots)', fontsize=12, fontweight='bold')
ax2.set_title('MAE Across Cross-Validation Folds', fontsize=13, fontweight='bold')
ax2.set_xticks(fold_numbers)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='best', fontsize=10)

plt.tight_layout()
plt.savefig('../results/cv_fold_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Cross-validation fold performance plot saved!")

In [ ]:
# 4. Predicted vs Actual Values (for best model - XGBoost on last fold)
print("Generating Predicted vs Actual plot for XGBoost...")
fold_idx = 0
for train_idx, test_idx in kfold.split(X_scaled):
    if fold_idx == 4:  # Use last fold
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        xgb_final = xgb.XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
        )
        xgb_final.fit(X_train, y_train)
        y_pred_final = xgb_final.predict(X_test)
        break
    fold_idx += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Predicted vs Actual
ax1.scatter(y_test, y_pred_final, alpha=0.5, s=30, color='steelblue', edgecolor='navy', linewidth=0.5)
min_val = min(y_test.min(), y_pred_final.min())
max_val = max(y_test.max(), y_pred_final.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Ship Speed (knots)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Predicted Ship Speed (knots)', fontsize=12, fontweight='bold')
ax1.set_title('XGBoost: Predicted vs Actual Ship Speed', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Residuals plot
residuals = y_test - y_pred_final
ax2.scatter(y_pred_final, residuals, alpha=0.5, s=30, color='coral', edgecolor='darkred', linewidth=0.5)
ax2.axhline(y=0, color='r', linestyle='--', lw=2)
ax2.set_xlabel('Predicted Ship Speed (knots)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Residuals (knots)', fontsize=12, fontweight='bold')
ax2.set_title('XGBoost: Residual Plot', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add statistics
mae_val = mean_absolute_error(y_test, y_pred_final)
rmse_val = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_val = r2_score(y_test, y_pred_final)

stats_text = f'MAE: {mae_val:.4f}\nRMSE: {rmse_val:.4f}\nR²: {r2_val:.4f}'
ax1.text(0.05, 0.95, stats_text, transform=ax1.transAxes, 
        fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('../results/predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

print("Predicted vs Actual plot saved!")


In [ ]:
# 5. Distribution of residuals
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of residuals
ax1.hist(residuals, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax1.set_xlabel('Residual (knots)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of XGBoost Residuals', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.legend()

# Q-Q plot (residuals normality)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot of Residuals (Normality Check)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/residuals_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Residuals distribution plot saved!")

## 11. Summary and Key Findings

In [ ]:
print("="*80)
print("MARITIME SPEED PREDICTION - BASELINE ML MODEL SUMMARY")
print("="*80)

print("\n📊 DATASET STATISTICS:")
print(f"  • Total samples: {len(df_clean):,}")
print(f"  • Number of features: {len(feature_cols)}")
print(f"  • Target variable: STW_kn (Ship Through Water Speed)")
print(f"  • Speed range: {df_clean[target].min():.2f} - {df_clean[target].max():.2f} knots")
print(f"  • Mean speed: {df_clean[target].mean():.2f} ± {df_clean[target].std():.2f} knots")

print("\n🎯 MODEL EVALUATION (5-Fold Cross-Validation):")
print("-" * 80)
for model_name, metrics in all_results.items():
    print(f"\n  {model_name}:")
    print(f"    • MAE:  {metrics['MAE_mean']:.4f} ± {metrics['MAE_std']:.4f} knots")
    print(f"    • RMSE: {metrics['RMSE_mean']:.4f} ± {metrics['RMSE_std']:.4f} knots")
    print(f"    • R²:   {metrics['R2_mean']:.4f} ± {metrics['R2_std']:.4f}")

print("\n⚡ BEST MODEL: XGBoost")
print(f"  • Average prediction error: ±{xgb_results['MAE_mean']:.3f} knots")
print(f"  • Explains {xgb_results['R2_mean']*100:.1f}% of speed variance")

print("\n🌊 TOP WEATHER FACTORS AFFECTING SHIP SPEED:")
print("-" * 80)
for idx, row in weather_importance_df.head(8).iterrows():
    print(f"  {idx+1}. {row['Feature']:20s} - Importance: {row['Importance']:.4f} ({row['Normalized']*100:.1f}%)")

print(f"\n  Total weather impact: {weather_importance_df['Importance'].sum()/sorted_importance.sum()*100:.1f}% of model")

print("\n📈 PERFORMANCE METRICS EXPLANATION:")
print("-" * 80)
print("  • MAE (Mean Absolute Error): Average deviation from true speed (in knots)")
print("    Lower is better. Our XGBoost ±{:.2f} means ~{:.0f}% accuracy on {:.1f} knot avg".format(
    xgb_results['MAE_mean'], 
    (1 - xgb_results['MAE_mean']/df_clean[target].mean())*100,
    df_clean[target].mean()))
print("  • RMSE (Root Mean Squared Error): Penalizes larger errors more heavily")
print(f"    Our XGBoost: {xgb_results['RMSE_mean']:.4f} knots")
print("  • R² Score: Proportion of variance explained (0-1, higher is better)")
print(f"    Our XGBoost: {xgb_results['R2_mean']:.4f} (predicts {xgb_results['R2_mean']*100:.1f}% of speed variation)")
print("  • Accuracy (custom): % of predictions within ±1 knot tolerance")
print("    Suitable for maritime operations where small deviations are acceptable")

print("\n" + "="*80)
print("✅ BASELINE ML MODEL COMPLETE - Ready for next iteration")
print("="*80)